In [147]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
plt.style.use('ggplot')

In [148]:
path = '../data/dataset/dataset.csv'
# path = '../data/processed_data/daily_average_all.csv'
data = pd.read_csv(path)
data.head()

,date,pm,location,temp,dew_point,humidity,pressure,wind_speed,precipitation,wind_dir_sin,wind_dir_cos,month,month_sin,month_cos
0,2023-01-01,104.875000,DN,23.5,16.9,68,1015.4,7.1,0.0,0.017452,0.999848,1,0.5,0.866025
1,2023-01-01,79.625000,BN,23.7,16.1,65,1015.3,6.7,0.0,0.707107,0.707107,1,0.5,0.866025
2,2023-01-01,79.375000,KPS,22.9,16.1,68,1015.6,7.2,0.0,0.325568,0.945519,1,0.5,0.866025
3,2023-01-02,90.583333,DN,24.0,17.4,69,1015.7,6.4,0.0,0.156434,0.987688,1,0.5,0.866025
4,2023-01-02,89.500000,BN,24.2,17.9,69,1015.6,5.3,0.0,0.642788,0.766044,1,0.5,0.866025


In [149]:
# data['date'] = pd.to_datetime(data['date'])
# data['month'] = data['date'].dt.month

def get_season(month):
    if month in [3, 4, 5]:
        return 'hot'
    elif month in [9, 10, 11]:
        return 'rain'
    elif month in [12, 1, 2]:
        return 'winter'

data['season'] = data['month'].apply(get_season)

data_encoded = pd.get_dummies(data, columns=['location', 'season'], drop_first=False)
data_encoded['original_location'] = data['location']
data_encoded = data_encoded.sort_values(by=['original_location', 'date']).reset_index(drop=True)
# data_encoded = data_encoded.drop(columns=['Unnamed: 0','index'])

data_encoded.head()

,date,pm,temp,dew_point,humidity,pressure,wind_speed,precipitation,wind_dir_sin,wind_dir_cos,month,month_sin,month_cos,location_BN,location_DN,location_KPS,season_hot,season_rain,season_winter,original_location
0,2023-01-01,79.625000,23.7,16.1,65,1015.3,6.7,0.0,0.707107,0.707107,1,0.5,0.866025,True,False,False,False,False,True,BN
1,2023-01-02,89.500000,24.2,17.9,69,1015.6,5.3,0.0,0.642788,0.766044,1,0.5,0.866025,True,False,False,False,False,True,BN
2,2023-01-03,110.500000,25.6,18.9,68,1014.7,6.1,0.0,0.515038,0.857167,1,0.5,0.866025,True,False,False,False,False,True,BN
3,2023-01-04,97.166667,26.5,19.1,65,1014.1,5.9,0.0,0.707107,0.707107,1,0.5,0.866025,True,False,False,False,False,True,BN
4,2023-01-05,95.583333,26.1,19.0,66,1013.7,6.3,0.0,0.681998,0.731354,1,0.5,0.866025,True,False,False,False,False,True,BN


In [150]:
def create_sliding_window(X, y, window_size=7):
    X_out, y_out = [], []
    for i in range(len(X) - window_size):
        window = X[i:i + window_size]
        X_out.append(window.flatten())
        y_out.append(y[i + window_size])
    return np.array(X_out), np.array(y_out)

window_size = 7

X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

for loc in data_encoded['original_location'].unique():
    
    df_loc = data_encoded[data_encoded['original_location'] == loc].copy()
    
    X_loc = df_loc.drop(columns=["date", "month", "original_location"
                                 ,"precipitation","humidity","wind_speed","location_DN","location_KPS","location_BN"]).values
    y_loc = df_loc["pm"].values
    
    X_seq, y_seq = create_sliding_window(X_loc, y_loc, window_size)
    
    n = len(X_seq)
    train_end = int(n * 0.8)
    val_end   = int(n * 0.9)
    
    X_train_list.append(X_seq[:train_end])
    y_train_list.append(y_seq[:train_end])
    
    X_val_list.append(X_seq[train_end:val_end])
    y_val_list.append(y_seq[train_end:val_end])
    
    X_test_list.append(X_seq[val_end:])
    y_test_list.append(y_seq[val_end:])

X_train = np.vstack(X_train_list)
y_train = np.concatenate(y_train_list)

X_val = np.vstack(X_val_list)
y_val = np.concatenate(y_val_list)

X_test = np.vstack(X_test_list)
y_test = np.concatenate(y_test_list)

print("Train shape:", X_train.shape, y_train.shape)
print("Val shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (2613, 77) (2613,)
Val shape: (327, 77) (327,)
Test shape: (327, 77) (327,)


In [151]:
def objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 100, 500)
    max_depth = trial.suggest_int("max_depth", 3, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)
    max_features = trial.suggest_categorical(
        "max_features", ["sqrt", "log2", None]
    )

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    return rmse


In [152]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)
print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)


[I 2026-02-17 04:16:53,050] A new study created in memory with name: no-name-c9ab4413-a187-4eff-abd6-d7d48b490e20
[I 2026-02-17 04:16:53,647] Trial 0 finished with value: 10.104333632786261 and parameters: {'n_estimators': 117, 'max_depth': 8, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': None}. Best is trial 0 with value: 10.104333632786261.
[I 2026-02-17 04:16:54,446] Trial 1 finished with value: 10.361573782254396 and parameters: {'n_estimators': 178, 'max_depth': 6, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 0 with value: 10.104333632786261.
[I 2026-02-17 04:16:55,818] Trial 2 finished with value: 10.257728445907864 and parameters: {'n_estimators': 291, 'max_depth': 6, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': None}. Best is trial 0 with value: 10.104333632786261.
[I 2026-02-17 04:16:56,178] Trial 3 finished with value: 12.03675967653902 and parameters: {'n_estimators': 184, 'max_depth': 11, 'min_samples

Best RMSE: 10.054788763962911
Best params: {'n_estimators': 326, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 5, 'max_features': None}


In [153]:
X_trainval = np.vstack((X_train, X_val))
y_trainval = np.concatenate((y_train, y_val))

best_params = study.best_params

best_rf = RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_trainval, y_trainval)

RandomForestRegressor(max_depth=10, max_features=None, min_samples_leaf=5,
                      min_samples_split=8, n_estimators=326, n_jobs=-1,
                      random_state=42)

In [154]:
def mean_bias_error(y_true, y_pred):
    return np.mean(y_pred - y_true)

In [155]:
y_pred = best_rf.predict(X_test)


In [156]:
r2   = r2_score(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
mbe  = mean_bias_error(y_test, y_pred)

print("===== Test Metrics =====")
print(f"R2   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MBE  : {mbe:.4f}")


===== Test Metrics =====
R2   : 0.4926
MSE  : 170.3517
MAE  : 10.1995
RMSE : 13.0519
MBE  : 3.5374
